In [1]:
#Install dependencies with uv (faster alternative to pip)
!uv pip install anthropic python-dotenv

Audited 2 packages in 18ms


In [2]:
# Helper classes for immutable messages and Anthropic client with context management

import anthropic

class ImmutableDict(dict):
    """
    A dictionary subclass that becomes immutable after initialization.
    Prevents any modification operations after the _immutable flag is set.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Set immutable flag AFTER initialization
        object.__setattr__(self, '_immutable', True)
    
    def __setitem__(self, key, value):
        """Prevent modification after initialization."""
        if getattr(self, '_immutable', False):
            raise TypeError(f"ImmutableDict is immutable - cannot modify '{key}'")
        super().__setitem__(key, value)
    
    def __delitem__(self, key):
        """Prevent deletion of items."""
        if getattr(self, '_immutable', False):
            raise TypeError("ImmutableDict is immutable - cannot delete items")
        super().__delitem__(key)
    
    def clear(self):
        """Prevent clearing the dictionary."""
        raise TypeError("ImmutableDict is immutable - cannot clear")
    
    def pop(self, key, default=None):
        """Prevent popping items."""
        raise TypeError("ImmutableDict is immutable - cannot pop items")
    
    def popitem(self):
        """Prevent popping items."""
        raise TypeError("ImmutableDict is immutable - cannot pop items")
    
    def setdefault(self, key, default=None):
        """Prevent setting defaults."""
        raise TypeError("ImmutableDict is immutable - cannot set defaults")
    
    def update(self, *args, **kwargs):
        """Prevent updating the dictionary."""
        raise TypeError("ImmutableDict is immutable - cannot update")

class AnthropicMessage(ImmutableDict):
    """
    Immutable message class for Anthropic API communication.
    Inherits immutability from ImmutableDict and adds API-specific properties.
    """
    def __init__(self, role: str, content: str):
        # Initialize with the data, then ImmutableDict will set _immutable flag
        super().__init__(role=role, content=content)
    
    @property
    def role(self) -> str:
        """Get the role of the message."""
        return self["role"]
    
    @property
    def content(self) -> str:
        """Get the content of the message."""
        return self["content"]

class UserMessage(AnthropicMessage):
    """
    Immutable helper class for user messages.
    """
    def __init__(self, content: str):
        super().__init__(role="user", content=content)
        
class AssistantMessage(AnthropicMessage):
    """
    Immutable helper class for assistant messages.
    """
    def __init__(self, content: str):
        super().__init__(role="assistant", content=content)

class AnthropicClient(anthropic.Anthropic):
    """
    Extended Anthropic client with model configuration and context management.
    """
    def __init__(self, api_key: str, model: str):
        super().__init__(api_key=api_key)
        self.model = model
        self._context: list[AnthropicMessage] = []
    
    @property
    def context(self) -> list[AnthropicMessage]:
        """Get the current context messages (read-only)."""
        return self._context
    
    def chat(self, message: str) -> str:
        """
        Send a message to Claude and get a response, managing context automatically.
        
        Args:
            message (str): The message to send to Claude
            
        Returns:
            str: Claude's response text
        """
        # Add user message to context
        user_message = UserMessage(content=message)
        self._context.append(user_message)
        
        print(f"📤 Sending {len(self._context)} messages to Claude")
        
        # No conversion needed! AnthropicMessage objects are already dictionaries
        response = self.messages.create(
            model=self.model,  # Using model from client configuration
            max_tokens=1000,
            temperature=0,
            messages=self._context  # Use client's own context
        )
        
        response_text = response.content[0].text
        
        # Add assistant response to context
        assistant_message = AssistantMessage(content=response_text)
        self._context.append(assistant_message)
        
        print("✅ API request successful!")
        return response_text
    
    def print_context(self):
        """
        Display the client's current context messages in a nice format.
        """
        if not self._context:
            print("📝 Context is empty")
            return
        
        print(f"📝 Context ({len(self._context)} messages):")
        print("-" * 50)
        
        for i, message in enumerate(self._context, 1):
            role_emoji = "👤" if message.role == "user" else "🤖"
            print(f"{i}. {role_emoji} {message.role.title()}: {message.content}")
        print("-" * 50)


print("✅ AnthropicClient class and helper functions defined")

✅ AnthropicClient class and helper functions defined


In [3]:
# Load env variables
import os
from dotenv import load_dotenv

load_dotenv()

# Get API key and model from environment
api_key = os.getenv('ANTHROPIC_API_KEY')
model = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-0')  # default fallback

if not api_key:
    print("Warning: ANTHROPIC_API_KEY not found in environment variables")
else:
    print("✅ API key loaded successfully")
    
print(f"✅ Using model: {model}")

✅ API key loaded successfully
✅ Using model: claude-haiku-4-5


In [4]:
# Create an API client

client = AnthropicClient(
    api_key=api_key,
    model=model
)

print("✅ Anthropic client created successfully")
print(f"✅ Client configured with model: {client.model}")

✅ Anthropic client created successfully
✅ Client configured with model: claude-haiku-4-5


In [5]:
# Pass the list of messages into the 'chat' to get an answer from Claude
message = "Define quantum computing in one sentence"
response = client.chat(message)
print(f"Message: {message}")
print(f"\nClaude's response: {response}")
print("\n" + "="*60)
client.print_context()

📤 Sending 1 messages to Claude
✅ API request successful!
Message: Define quantum computing in one sentence

Claude's response: Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, enabling exponentially faster solutions to certain complex problems.

📝 Context (2 messages):
--------------------------------------------------
1. 👤 User: Define quantum computing in one sentence
2. 🤖 Assistant: Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, enabling exponentially faster solutions to certain complex problems.
--------------------------------------------------


In [6]:
# Add in the user's follow-up question and send to claude again
message = "Write another sentence."
response = client.chat(message)
print(f"Message: {message}")
print(f"\nClaude's response: {response}")
print("\n" + "="*60)
client.print_context()

📤 Sending 3 messages to Claude
✅ API request successful!
Message: Write another sentence.

Claude's response: Unlike classical bits that exist as either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously, allowing quantum computers to explore many possible solutions in parallel.

📝 Context (4 messages):
--------------------------------------------------
1. 👤 User: Define quantum computing in one sentence
2. 🤖 Assistant: Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers, enabling exponentially faster solutions to certain complex problems.
3. 👤 User: Write another sentence.
4. 🤖 Assistant: Unlike classical bits that exist as either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously, allowing quantum computers to explore many possible solutions in parallel.
--------------------------------------------------
